# 🔍 Manufacturing Defect Detection SystemCNN-based visual inspection using transfer learning and Grad-CAM for explainable AI.

In [ ]:
# Install dependencies!pip install numpy pillow plotly -q

## 1. Configuration

In [ ]:
"""Configuration Module for Manufacturing Defect Detection SystemCentralizes all configuration parameters."""from pathlib import Path# ============================================================================# PATHS# ============================================================================PROJECT_ROOT = Path(__file__).parentDATA_DIR = PROJECT_ROOT / "data"TRAIN_DIR = DATA_DIR / "train"TEST_DIR = DATA_DIR / "test"MODELS_DIR = PROJECT_ROOT / "saved_models"for d in [DATA_DIR, TRAIN_DIR, TEST_DIR, MODELS_DIR]:    d.mkdir(parents=True, exist_ok=True)# ============================================================================# IMAGE CONFIGURATION# ============================================================================IMAGE_SIZE = (224, 224)  # Standard for EfficientNet/ResNetCHANNELS = 3BATCH_SIZE = 32# Normalization (ImageNet stats for transfer learning)IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD = [0.229, 0.224, 0.225]# ============================================================================# CLASS LABELS# ============================================================================CLASSES = ["good", "defect"]NUM_CLASSES = 2# Defect subtypes (for synthetic data generation)DEFECT_TYPES = ["scratch", "crack", "contamination", "dent"]# ============================================================================# MODEL CONFIGURATION# ============================================================================MODEL_NAME = "efficientnet_b0"  # Options: resnet18, efficientnet_b0PRETRAINED = TrueFREEZE_BACKBONE = False  # Fine-tune all layers# Training hyperparametersEPOCHS = 20LEARNING_RATE = 0.001WEIGHT_DECAY = 1e-4EARLY_STOPPING_PATIENCE = 5# ============================================================================# INFERENCE CONFIGURATION# ============================================================================CONFIDENCE_THRESHOLD = 0.5  # Above this = defectHIGH_CONFIDENCE_THRESHOLD = 0.8  # High confidence detection# ============================================================================# GRAD-CAM CONFIGURATION# ============================================================================GRADCAM_LAYER = "features"  # Target layer for activation mapsHEATMAP_ALPHA = 0.4  # Overlay transparency# ============================================================================# EVALUATION METRICS# ============================================================================ACCURACY_TARGET = 0.95RECALL_TARGET = 0.98  # Critical: don't miss defectsPRECISION_TARGET = 0.90  # Minimize false alarms# ============================================================================# APPLICATION CONFIGURATION# ============================================================================APP_TITLE = "🔍 Manufacturing Defect Detection"APP_LAYOUT = "wide"DEBUG_MODE = True# ============================================================================# SYNTHETIC DATA CONFIGURATION# ============================================================================SYNTHETIC_TRAIN_SAMPLES = 400  # 200 good + 200 defectSYNTHETIC_TEST_SAMPLES = 100   # 50 good + 50 defectSYNTHETIC_IMAGE_SIZE = 224

## 2. Preprocessing & Synthetic Data

In [ ]:
"""Image Preprocessing and Data GenerationHandles:1. Synthetic defect image generation2. Image preprocessing and augmentation3. Data loading utilities"""import numpy as npfrom PIL import Image, ImageDraw, ImageFilterimport randomfrom pathlib import Pathfrom typing import Tuple, Listimport loggingfrom config import (    TRAIN_DIR, TEST_DIR, IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD,    SYNTHETIC_TRAIN_SAMPLES, SYNTHETIC_TEST_SAMPLES, SYNTHETIC_IMAGE_SIZE,    DEFECT_TYPES)logging.basicConfig(level=logging.INFO)logger = logging.getLogger(__name__)def create_base_surface(size: int = SYNTHETIC_IMAGE_SIZE) -> Image.Image:    """Create a synthetic metal/plastic surface."""    # Random base color (metallic gray to silver)    base_color = random.randint(160, 200)    img = Image.new('RGB', (size, size), (base_color, base_color, base_color))        # Add subtle texture    pixels = np.array(img)    noise = np.random.normal(0, 5, pixels.shape).astype(np.int16)    pixels = np.clip(pixels.astype(np.int16) + noise, 0, 255).astype(np.uint8)    img = Image.fromarray(pixels)        # Add slight gradient for realism    draw = ImageDraw.Draw(img)    for i in range(0, size, 10):        alpha = int(10 * np.sin(i / size * np.pi))        draw.line([(0, i), (size, i)], fill=(base_color + alpha, base_color + alpha, base_color + alpha))        return imgdef add_scratch(img: Image.Image) -> Image.Image:    """Add scratch defect to image."""    draw = ImageDraw.Draw(img)    size = img.size[0]        # Random scratch parameters    num_scratches = random.randint(1, 3)        for _ in range(num_scratches):        x1 = random.randint(10, size - 10)        y1 = random.randint(10, size - 10)                # Scratch direction and length        angle = random.uniform(0, 2 * np.pi)        length = random.randint(30, 100)                x2 = int(x1 + length * np.cos(angle))        y2 = int(y1 + length * np.sin(angle))                # Dark scratch line        scratch_color = random.randint(30, 80)        width = random.randint(1, 3)        draw.line([(x1, y1), (x2, y2)], fill=(scratch_color, scratch_color, scratch_color), width=width)        return imgdef add_crack(img: Image.Image) -> Image.Image:    """Add crack defect to image."""    draw = ImageDraw.Draw(img)    size = img.size[0]        # Start point    x, y = random.randint(20, size - 20), random.randint(20, size - 20)        # Generate crack path (random walk)    points = [(x, y)]    for _ in range(random.randint(5, 10)):        dx = random.randint(-20, 20)        dy = random.randint(-20, 20)        x = max(5, min(size - 5, x + dx))        y = max(5, min(size - 5, y + dy))        points.append((x, y))        # Draw crack    crack_color = random.randint(20, 60)    draw.line(points, fill=(crack_color, crack_color, crack_color), width=1)        # Add branches    for i in range(0, len(points) - 1, 2):        if random.random() > 0.5:            bx = points[i][0] + random.randint(-15, 15)            by = points[i][1] + random.randint(-15, 15)            draw.line([points[i], (bx, by)], fill=(crack_color, crack_color, crack_color), width=1)        return imgdef add_contamination(img: Image.Image) -> Image.Image:    """Add contamination spots to image."""    draw = ImageDraw.Draw(img)    size = img.size[0]        # Random spots    num_spots = random.randint(2, 6)        for _ in range(num_spots):        cx = random.randint(20, size - 20)        cy = random.randint(20, size - 20)        radius = random.randint(5, 20)                # Dark or discolored spot        if random.random() > 0.5:            spot_color = (random.randint(40, 80), random.randint(40, 80), random.randint(40, 80))        else:            spot_color = (random.randint(100, 140), random.randint(80, 100), random.randint(60, 80))                draw.ellipse([cx - radius, cy - radius, cx + radius, cy + radius], fill=spot_color)        return imgdef add_dent(img: Image.Image) -> Image.Image:    """Add dent mark to image."""    draw = ImageDraw.Draw(img)    size = img.size[0]        # Dent center    cx = random.randint(30, size - 30)    cy = random.randint(30, size - 30)    radius = random.randint(15, 40)        # Create dent effect (lighter center, darker edge)    for r in range(radius, 0, -2):        alpha = int(30 * (1 - r / radius))        color = 180 + alpha        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(color, color, color))        # Dark shadow on one side    shadow_offset = 5    shadow_color = 120    draw.arc(        [cx - radius + shadow_offset, cy - radius + shadow_offset,          cx + radius + shadow_offset, cy + radius + shadow_offset],        start=45, end=225, fill=(shadow_color, shadow_color, shadow_color), width=2    )        return imgdef generate_defect_image(defect_type: str = None) -> Image.Image:    """Generate a synthetic defect image."""    img = create_base_surface()        if defect_type is None:        defect_type = random.choice(DEFECT_TYPES)        if defect_type == "scratch":        img = add_scratch(img)    elif defect_type == "crack":        img = add_crack(img)    elif defect_type == "contamination":        img = add_contamination(img)    elif defect_type == "dent":        img = add_dent(img)        return imgdef generate_good_image() -> Image.Image:    """Generate a synthetic non-defective image."""    return create_base_surface()def generate_synthetic_dataset(    train_samples: int = SYNTHETIC_TRAIN_SAMPLES,    test_samples: int = SYNTHETIC_TEST_SAMPLES) -> None:    """Generate full synthetic dataset."""        logger.info("Generating synthetic defect detection dataset...")        # Training data    train_good = train_samples // 2    train_defect = train_samples // 2        # Generate training good images    good_dir = TRAIN_DIR / "good"    good_dir.mkdir(parents=True, exist_ok=True)    for i in range(train_good):        img = generate_good_image()        img.save(good_dir / f"good_{i:04d}.png")    logger.info(f"Generated {train_good} training good images")        # Generate training defect images    defect_dir = TRAIN_DIR / "defect"    defect_dir.mkdir(parents=True, exist_ok=True)    for i in range(train_defect):        defect_type = DEFECT_TYPES[i % len(DEFECT_TYPES)]        img = generate_defect_image(defect_type)        img.save(defect_dir / f"defect_{defect_type}_{i:04d}.png")    logger.info(f"Generated {train_defect} training defect images")        # Test data    test_good = test_samples // 2    test_defect = test_samples // 2        # Generate test good images    test_good_dir = TEST_DIR / "good"    test_good_dir.mkdir(parents=True, exist_ok=True)    for i in range(test_good):        img = generate_good_image()        img.save(test_good_dir / f"good_{i:04d}.png")    logger.info(f"Generated {test_good} test good images")        # Generate test defect images    test_defect_dir = TEST_DIR / "defect"    test_defect_dir.mkdir(parents=True, exist_ok=True)    for i in range(test_defect):        defect_type = DEFECT_TYPES[i % len(DEFECT_TYPES)]        img = generate_defect_image(defect_type)        img.save(test_defect_dir / f"defect_{defect_type}_{i:04d}.png")    logger.info(f"Generated {test_defect} test defect images")        logger.info("Dataset generation complete!")def preprocess_image(    image: Image.Image,    size: Tuple[int, int] = IMAGE_SIZE) -> np.ndarray:    """    Preprocess image for model input.        Returns:        Normalized numpy array (C, H, W)    """    # Resize    img = image.resize(size, Image.BILINEAR)        # Convert to numpy    img_array = np.array(img).astype(np.float32) / 255.0        # Normalize with ImageNet stats    mean = np.array(IMAGENET_MEAN)    std = np.array(IMAGENET_STD)    img_array = (img_array - mean) / std        # HWC to CHW    img_array = np.transpose(img_array, (2, 0, 1))        return img_arraydef load_image(path: str) -> Image.Image:    """Load an image from path."""    return Image.open(path).convert('RGB')if __name__ == "__main__":    generate_synthetic_dataset()

## 3. Generate Dataset

In [ ]:
from models.preprocessing import generate_synthetic_dataset, generate_defect_image, generate_good_image# Generate datasetgenerate_synthetic_dataset()# Show samplesimport matplotlib.pyplot as pltfig, axes = plt.subplots(1, 4, figsize=(16, 4))axes[0].imshow(generate_good_image())axes[0].set_title('Good')axes[1].imshow(generate_defect_image('scratch'))axes[1].set_title('Scratch')axes[2].imshow(generate_defect_image('crack'))axes[2].set_title('Crack')axes[3].imshow(generate_defect_image('contamination'))axes[3].set_title('Contamination')plt.show()

## 4. CNN Classifier

In [ ]:
"""CNN Classifier for Defect DetectionImplements:1. Transfer learning with EfficientNet/ResNet2. Binary classification (Good vs Defect)3. Training and evaluation"""import numpy as npfrom typing import Tuple, List, Optional, Dictimport loggingfrom pathlib import Pathfrom config import (    NUM_CLASSES, CLASSES, IMAGE_SIZE, EPOCHS, LEARNING_RATE,    IMAGENET_MEAN, IMAGENET_STD, MODELS_DIR, CONFIDENCE_THRESHOLD)logger = logging.getLogger(__name__)# Check for PyTorch availabilitytry:    import torch    import torch.nn as nn    import torch.optim as optim    from torch.utils.data import Dataset, DataLoader    import torchvision.models as models    import torchvision.transforms as transforms    TORCH_AVAILABLE = Trueexcept ImportError:    TORCH_AVAILABLE = False    logger.warning("PyTorch not available. Using simple classifier fallback.")class SimpleClassifier:    """    Simple baseline classifier using color histogram features.    Used when PyTorch is not available.    """        def __init__(self):        self.threshold = 0.3  # Defect detection threshold        self.is_fitted = True        def predict(self, image: np.ndarray) -> Tuple[str, float]:        """        Simple defect detection based on color variance.                Args:            image: RGB image array (H, W, 3) in range [0, 255]                    Returns:            (prediction, confidence)        """        # Calculate standard deviation of pixel values        std = np.std(image)                # Higher variance often indicates defects        defect_score = min(std / 50.0, 1.0)  # Normalize                if defect_score > self.threshold:            return "defect", defect_score        else:            return "good", 1.0 - defect_score        def predict_proba(self, image: np.ndarray) -> np.ndarray:        """Return class probabilities."""        _, score = self.predict(image)        if score > 0.5:            return np.array([1 - score, score])        else:            return np.array([score, 1 - score])if TORCH_AVAILABLE:    class DefectDataset(Dataset):        """PyTorch Dataset for defect images."""                def __init__(            self,            root_dir: str,            transform=None        ):            self.root_dir = Path(root_dir)            self.transform = transform            self.samples = []                        # Load good images            good_dir = self.root_dir / "good"            if good_dir.exists():                for img_path in good_dir.glob("*.png"):                    self.samples.append((str(img_path), 0))                for img_path in good_dir.glob("*.jpg"):                    self.samples.append((str(img_path), 0))                        # Load defect images            defect_dir = self.root_dir / "defect"            if defect_dir.exists():                for img_path in defect_dir.glob("*.png"):                    self.samples.append((str(img_path), 1))                for img_path in defect_dir.glob("*.jpg"):                    self.samples.append((str(img_path), 1))                        logger.info(f"Loaded {len(self.samples)} images from {root_dir}")                def __len__(self):            return len(self.samples)                def __getitem__(self, idx):            from PIL import Image                        img_path, label = self.samples[idx]            image = Image.open(img_path).convert('RGB')                        if self.transform:                image = self.transform(image)                        return image, label            class DefectClassifier(nn.Module):        """        CNN for defect classification using transfer learning.        """                def __init__(            self,            model_name: str = "resnet18",            num_classes: int = NUM_CLASSES,            pretrained: bool = True        ):            super().__init__()                        self.model_name = model_name                        if model_name == "resnet18":                self.backbone = models.resnet18(                    weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None                )                num_features = self.backbone.fc.in_features                self.backbone.fc = nn.Linear(num_features, num_classes)                            elif model_name == "efficientnet_b0":                self.backbone = models.efficientnet_b0(                    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None                )                num_features = self.backbone.classifier[1].in_features                self.backbone.classifier[1] = nn.Linear(num_features, num_classes)                        else:                raise ValueError(f"Unknown model: {model_name}")                def forward(self, x: torch.Tensor) -> torch.Tensor:            return self.backbone(x)                def get_features_layer(self):            """Get the layer to use for Grad-CAM."""            if self.model_name == "resnet18":                return self.backbone.layer4            elif self.model_name == "efficientnet_b0":                return self.backbone.features[-1]class DefectDetector:    """    High-level wrapper for defect detection.        Handles model loading, inference, and training.    """        def __init__(        self,        model_name: str = "resnet18",        pretrained: bool = True,        device: str = None    ):        self.model_name = model_name        self.is_fitted = False                if TORCH_AVAILABLE:            self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')            self.model = DefectClassifier(model_name, pretrained=pretrained)            self.model = self.model.to(self.device)                        self.transform = transforms.Compose([                transforms.Resize(IMAGE_SIZE),                transforms.ToTensor(),                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)            ])                        self.train_transform = transforms.Compose([                transforms.Resize(IMAGE_SIZE),                transforms.RandomHorizontalFlip(),                transforms.RandomVerticalFlip(),                transforms.RandomRotation(15),                transforms.ColorJitter(brightness=0.2, contrast=0.2),                transforms.ToTensor(),                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)            ])        else:            self.model = SimpleClassifier()            self.is_fitted = True        def train(        self,        train_dir: str,        val_dir: str = None,        epochs: int = EPOCHS,        learning_rate: float = LEARNING_RATE,        batch_size: int = 32    ) -> Dict[str, List[float]]:        """        Train the model on the given dataset.                Returns:            Training history        """        if not TORCH_AVAILABLE:            logger.warning("PyTorch not available. Using pretrained fallback.")            return {"loss": [], "accuracy": []}                # Create datasets        train_dataset = DefectDataset(train_dir, transform=self.train_transform)        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)                if val_dir:            val_dataset = DefectDataset(val_dir, transform=self.transform)            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)                # Loss and optimizer        criterion = nn.CrossEntropyLoss()        optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)                history = {"loss": [], "accuracy": []}                self.model.train()        for epoch in range(epochs):            total_loss = 0            correct = 0            total = 0                        for images, labels in train_loader:                images = images.to(self.device)                labels = labels.to(self.device)                                optimizer.zero_grad()                outputs = self.model(images)                loss = criterion(outputs, labels)                loss.backward()                optimizer.step()                                total_loss += loss.item()                _, predicted = outputs.max(1)                total += labels.size(0)                correct += predicted.eq(labels).sum().item()                        epoch_loss = total_loss / len(train_loader)            epoch_acc = correct / total            history["loss"].append(epoch_loss)            history["accuracy"].append(epoch_acc)                        logger.info(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")                self.is_fitted = True        return history        def predict(        self,        image,        return_proba: bool = False    ) -> Tuple[str, float]:        """        Predict defect status for an image.                Args:            image: PIL Image or numpy array            return_proba: If True, return class probabilities                    Returns:            (class_name, confidence) or probabilities        """        from PIL import Image                if not TORCH_AVAILABLE:            if isinstance(image, Image.Image):                image = np.array(image)            return self.model.predict(image)                self.model.eval()                # Convert to PIL if needed        if isinstance(image, np.ndarray):            image = Image.fromarray(image.astype(np.uint8))                # Preprocess        img_tensor = self.transform(image).unsqueeze(0).to(self.device)                with torch.no_grad():            outputs = self.model(img_tensor)            proba = torch.softmax(outputs, dim=1).cpu().numpy()[0]                if return_proba:            return proba                pred_class = int(proba.argmax())        confidence = float(proba[pred_class])                return CLASSES[pred_class], confidence        def save(self, path: str = None):        """Save model weights."""        if not TORCH_AVAILABLE:            return                path = path or str(MODELS_DIR / "defect_detector.pth")        torch.save(self.model.state_dict(), path)        logger.info(f"Model saved to {path}")        def load(self, path: str = None):        """Load model weights."""        if not TORCH_AVAILABLE:            return                path = path or str(MODELS_DIR / "defect_detector.pth")        if Path(path).exists():            self.model.load_state_dict(torch.load(path, map_location=self.device))            self.is_fitted = True            logger.info(f"Model loaded from {path}")

## 5. Grad-CAM Localization

In [ ]:
"""Grad-CAM for Defect LocalizationGradient-weighted Class Activation Mapping to visualizewhich image regions triggered the defect detection."""import numpy as npfrom PIL import Imagefrom typing import Tuple, Optionalimport loggingfrom config import IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD, HEATMAP_ALPHAlogger = logging.getLogger(__name__)# Check for PyTorchtry:    import torch    import torch.nn.functional as F    TORCH_AVAILABLE = Trueexcept ImportError:    TORCH_AVAILABLE = Falsedef apply_colormap(heatmap: np.ndarray) -> np.ndarray:    """    Convert grayscale heatmap to RGB colormap.        Args:        heatmap: Normalized heatmap (0-1)            Returns:        RGB colormap array    """    # Jet-like colormap    heatmap = np.clip(heatmap, 0, 1)        # Simple jet colormap implementation    r = np.clip(4 * heatmap - 1.5, 0, 1)    g = np.clip(1 - 4 * np.abs(heatmap - 0.5), 0, 1)    b = np.clip(1.5 - 4 * heatmap, 0, 1)        colormap = np.stack([r, g, b], axis=-1)    return (colormap * 255).astype(np.uint8)class SimpleGradCAM:    """    Simplified Grad-CAM that works with simple heuristics.    Used when PyTorch is not available.    """        def generate(self, image: np.ndarray, prediction: str) -> np.ndarray:        """        Generate a simple heatmap based on intensity variance.        """        if len(image.shape) == 3:            gray = np.mean(image, axis=2)        else:            gray = image                # Region with highest variance is likely defect        from scipy.ndimage import uniform_filter                try:            local_mean = uniform_filter(gray, size=20)            local_sqmean = uniform_filter(gray ** 2, size=20)            local_var = np.sqrt(np.abs(local_sqmean - local_mean ** 2))        except ImportError:            # Fallback without scipy            local_var = np.abs(gray - np.mean(gray))                # Normalize        heatmap = (local_var - local_var.min()) / (local_var.max() - local_var.min() + 1e-8)                return heatmapif TORCH_AVAILABLE:    class GradCAM:        """        Grad-CAM implementation for CNN models.                Generates heatmaps showing which regions contributed most        to the classification decision.        """                def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):            self.model = model            self.target_layer = target_layer                        self.gradients = None            self.activations = None                        # Register hooks            self._register_hooks()                def _register_hooks(self):            """Register forward and backward hooks."""                        def forward_hook(module, input, output):                self.activations = output.detach()                        def backward_hook(module, grad_input, grad_output):                self.gradients = grad_output[0].detach()                        self.target_layer.register_forward_hook(forward_hook)            self.target_layer.register_full_backward_hook(backward_hook)                def generate(            self,            input_tensor: torch.Tensor,            target_class: int = None        ) -> np.ndarray:            """            Generate Grad-CAM heatmap.                        Args:                input_tensor: Preprocessed input (1, C, H, W)                target_class: Class to generate heatmap for (default: predicted)                            Returns:                Heatmap as numpy array (H, W)            """            self.model.eval()                        # Forward pass            output = self.model(input_tensor)                        if target_class is None:                target_class = output.argmax(dim=1).item()                        # Backward pass            self.model.zero_grad()            target = output[0, target_class]            target.backward()                        # Compute weights            gradients = self.gradients[0]  # (C, H, W)            activations = self.activations[0]  # (C, H, W)                        weights = gradients.mean(dim=(1, 2))  # (C,)                        # Weighted combination            cam = torch.zeros(activations.shape[1:], device=activations.device)            for i, w in enumerate(weights):                cam += w * activations[i]                        # ReLU and normalize            cam = F.relu(cam)            cam = cam - cam.min()            cam = cam / (cam.max() + 1e-8)                        # Resize to input size            cam = cam.unsqueeze(0).unsqueeze(0)            cam = F.interpolate(cam, size=IMAGE_SIZE, mode='bilinear', align_corners=False)            cam = cam.squeeze().cpu().numpy()                        return camdef create_heatmap_overlay(    image: Image.Image,    heatmap: np.ndarray,    alpha: float = HEATMAP_ALPHA) -> Image.Image:    """    Overlay heatmap on original image.        Args:        image: Original PIL Image        heatmap: Normalized heatmap (0-1)        alpha: Overlay transparency            Returns:        Combined image with heatmap overlay    """    # Resize heatmap to match image    img_array = np.array(image)        if heatmap.shape[:2] != img_array.shape[:2]:        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))        heatmap_img = heatmap_img.resize(image.size, Image.BILINEAR)        heatmap = np.array(heatmap_img) / 255.0        # Apply colormap    colormap = apply_colormap(heatmap)        # Blend    overlay = (1 - alpha) * img_array + alpha * colormap    overlay = np.clip(overlay, 0, 255).astype(np.uint8)        return Image.fromarray(overlay)class DefectLocalizer:    """    High-level interface for defect localization using Grad-CAM.    """        def __init__(self, detector):        """        Args:            detector: DefectDetector instance        """        self.detector = detector                if TORCH_AVAILABLE and hasattr(detector, 'model') and hasattr(detector.model, 'get_features_layer'):            target_layer = detector.model.get_features_layer()            self.gradcam = GradCAM(detector.model, target_layer)        else:            self.gradcam = SimpleGradCAM()        def localize(        self,        image: Image.Image,        target_class: int = None    ) -> Tuple[np.ndarray, Image.Image]:        """        Generate defect localization heatmap.                Returns:            (heatmap, overlay_image)        """        if TORCH_AVAILABLE and hasattr(self.detector, 'transform'):            # Prepare input            img_tensor = self.detector.transform(image).unsqueeze(0)            img_tensor = img_tensor.to(self.detector.device)                        # Generate heatmap            heatmap = self.gradcam.generate(img_tensor, target_class)        else:            img_array = np.array(image)            prediction, _ = self.detector.predict(image)            heatmap = self.gradcam.generate(img_array, prediction)                # Create overlay        overlay = create_heatmap_overlay(image, heatmap)                return heatmap, overlay

## 6. Evaluation Metrics

In [ ]:
"""Evaluation Metrics for Defect DetectionStandard classification metrics:- Accuracy- Precision (minimize false alarms)- Recall (don't miss defects)- F1 Score- Confusion Matrix"""import numpy as npfrom typing import Dict, List, Tuplefrom collections import Counterimport loggingfrom config import CLASSES, ACCURACY_TARGET, PRECISION_TARGET, RECALL_TARGETlogger = logging.getLogger(__name__)def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:    """Calculate accuracy."""    return np.mean(y_true == y_pred)def precision(y_true: np.ndarray, y_pred: np.ndarray, pos_label: int = 1) -> float:    """    Precision = TP / (TP + FP)        For defect detection: minimize false alarms    """    tp = np.sum((y_pred == pos_label) & (y_true == pos_label))    fp = np.sum((y_pred == pos_label) & (y_true != pos_label))        if tp + fp == 0:        return 0.0        return tp / (tp + fp)def recall(y_true: np.ndarray, y_pred: np.ndarray, pos_label: int = 1) -> float:    """    Recall = TP / (TP + FN)        For defect detection: don't miss defects (critical!)    """    tp = np.sum((y_pred == pos_label) & (y_true == pos_label))    fn = np.sum((y_pred != pos_label) & (y_true == pos_label))        if tp + fn == 0:        return 0.0        return tp / (tp + fn)def f1_score(y_true: np.ndarray, y_pred: np.ndarray, pos_label: int = 1) -> float:    """F1 = 2 * (Precision * Recall) / (Precision + Recall)"""    p = precision(y_true, y_pred, pos_label)    r = recall(y_true, y_pred, pos_label)        if p + r == 0:        return 0.0        return 2 * (p * r) / (p + r)def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:    """    Build confusion matrix.        Returns:        2x2 matrix [[TN, FP], [FN, TP]]    """    tn = np.sum((y_pred == 0) & (y_true == 0))    fp = np.sum((y_pred == 1) & (y_true == 0))    fn = np.sum((y_pred == 0) & (y_true == 1))    tp = np.sum((y_pred == 1) & (y_true == 1))        return np.array([[tn, fp], [fn, tp]])def specificity(y_true: np.ndarray, y_pred: np.ndarray) -> float:    """    Specificity = TN / (TN + FP)        True negative rate (correctly identifying good items)    """    tn = np.sum((y_pred == 0) & (y_true == 0))    fp = np.sum((y_pred == 1) & (y_true == 0))        if tn + fp == 0:        return 0.0        return tn / (tn + fp)def false_positive_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:    """FPR = FP / (FP + TN) = 1 - Specificity"""    return 1 - specificity(y_true, y_pred)def false_negative_rate(y_true: np.ndarray, y_pred: np.ndarray) -> float:    """FNR = FN / (FN + TP) = 1 - Recall"""    return 1 - recall(y_true, y_pred)def evaluate_classifier(    y_true: np.ndarray,    y_pred: np.ndarray,    y_proba: np.ndarray = None) -> Dict[str, float]:    """    Calculate all metrics for a classifier.        Args:        y_true: Ground truth labels        y_pred: Predicted labels        y_proba: Predicted probabilities (optional, for AUC)            Returns:        Dictionary of metric names to values    """    metrics = {        'Accuracy': accuracy(y_true, y_pred),        'Precision': precision(y_true, y_pred),        'Recall': recall(y_true, y_pred),        'F1 Score': f1_score(y_true, y_pred),        'Specificity': specificity(y_true, y_pred),        'False Positive Rate': false_positive_rate(y_true, y_pred),        'False Negative Rate': false_negative_rate(y_true, y_pred)    }        return metricsdef check_targets(metrics: Dict[str, float]) -> Dict[str, Tuple[bool, float, float]]:    """    Check if metrics meet targets.        Returns:        Dict[metric_name, (passed, actual, target)]    """    targets = {        'Accuracy': ACCURACY_TARGET,        'Precision': PRECISION_TARGET,        'Recall': RECALL_TARGET    }        results = {}    for metric, target in targets.items():        actual = metrics.get(metric, 0)        passed = actual >= target        results[metric] = (passed, actual, target)        return resultsclass DefectEvaluator:    """Evaluate defect detection model."""        def __init__(self):        self.results = {}        self.predictions = []        self.ground_truth = []        def add_prediction(self, y_true: int, y_pred: int):        """Add a single prediction."""        self.ground_truth.append(y_true)        self.predictions.append(y_pred)        def add_batch(self, y_true: np.ndarray, y_pred: np.ndarray):        """Add batch predictions."""        self.ground_truth.extend(y_true.tolist())        self.predictions.extend(y_pred.tolist())        def evaluate(self) -> Dict[str, float]:        """Calculate metrics from accumulated predictions."""        y_true = np.array(self.ground_truth)        y_pred = np.array(self.predictions)                self.results = evaluate_classifier(y_true, y_pred)        return self.results        def get_confusion_matrix(self) -> np.ndarray:        """Get confusion matrix."""        y_true = np.array(self.ground_truth)        y_pred = np.array(self.predictions)        return confusion_matrix(y_true, y_pred)        def summary(self) -> str:        """Generate summary report."""        if not self.results:            self.evaluate()                lines = ["Defect Detection Evaluation", "=" * 40]                for metric, value in self.results.items():            lines.append(f"{metric}: {value:.4f}")                # Target check        lines.append("\nTarget Check:")        targets = check_targets(self.results)        for metric, (passed, actual, target) in targets.items():            status = "✅ PASS" if passed else "❌ FAIL"            lines.append(f"  {metric}: {actual:.2%} vs {target:.2%} {status}")                # Confusion matrix        cm = self.get_confusion_matrix()        lines.append(f"\nConfusion Matrix:")        lines.append(f"  TN: {cm[0,0]}, FP: {cm[0,1]}")        lines.append(f"  FN: {cm[1,0]}, TP: {cm[1,1]}")                return "\n".join(lines)        def reset(self):        """Reset evaluator."""        self.results = {}        self.predictions = []        self.ground_truth = []

## 7. Run Inference

In [ ]:
from PIL import Imagefrom models.classifier import DefectDetectorfrom models.gradcam import DefectLocalizer# Initialize detectordetector = DefectDetector(model_name='resnet18', pretrained=True)# Generate test imagetest_img = generate_defect_image('scratch')# Predictprediction, confidence = detector.predict(test_img)print(f'Prediction: {prediction}')print(f'Confidence: {confidence:.1%}')